# Nuclear binding energy

*nuclear · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/binding_energy)

The semi-empirical (liquid-drop) mass formula gives the binding energy of every nucleus from five terms: volume, surface, Coulomb, asymmetry and pairing. Evaluating it on a whole $(Z, N)$ grid at once by broadcasting shows why iron and nickel are the most tightly bound and where the valley of stability runs.

**Vector form**

$$B(Z,N) = a_V A - a_S A^{2/3} - a_C\,\frac{Z(Z-1)}{A^{1/3}} - a_A\,\frac{(N-Z)^2}{A} + \delta(Z,N),\qquad A = Z + N$$

**Index (tensor) form**

$$B_{zn} = a_V A_{zn} - a_S A_{zn}^{2/3} - a_C\,\frac{Z_z(Z_z-1)}{A_{zn}^{1/3}} - a_A\,\frac{(N_n-Z_z)^2}{A_{zn}} + \delta_{zn},\qquad A_{zn} = Z_z + N_n$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $Z_z,\ N_n$ | `grid(z_max, n_max)` | (Z, 1), (1, N) | proton numbers down a column, neutron numbers along a row |
| $A_{zn}$ | `z + n` | (Z, N) | broadcasting a column plus a row fills the whole table |
| $\delta_{zn}$ | `pairing(z, n, a_P)` | (Z, N) | $+a_P/\sqrt A$ even-even, $-a_P/\sqrt A$ odd-odd, 0 for odd $A$ |
| $B_{zn}$ | `binding_energy(z, n, coeff)` | (Z, N) | binding energy of every nucleus on the chart, MeV |
| $B_{zn}/A_{zn}$ | `per_nucleon(b, z, n)` | (Z, N) | binding energy per nucleon (heatmap) |
| $\max_{z} B_{zn}\big\vert _{A}$ | `most_stable(b, z, n)` | (K,) | most bound isobar for each mass number (red line) |
| $Z_0(A)$ | `stability_line(mass, coeff)` | (K,) | $A/(2 + \tfrac{a_C}{2a_A}A^{2/3})$ from $\partial B/\partial Z = 0$ |

Volume binding grows like $A$, but surface nucleons have fewer neighbours ($-A^{2/3}$) and the proton repulsion grows like $Z^2/A^{1/3}$; the balance puts the maximum of $B/A$ near $A \approx 56$–$62$, which is why fusion of light nuclei and fission of heavy ones both release energy. The asymmetry term pulls nuclei towards $N = Z$, the Coulomb term pushes heavy nuclei to $N > Z$: together they give the curved valley of stability $Z_0 \approx A/(2 + 0.0154\,A^{2/3})$. The formula is good to about 1 % for medium and heavy nuclei but misses the very light ones (⁴He is far more bound than a liquid drop) and the shell effects at magic numbers. The default coefficients are a common least-squares fit; changing them shows which term shapes which feature.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/binding_energy/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import binding_energy, per_nucleon, stability_line


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    coeff = tensors.coefficients(params)
    z, n = tensors.grid(params["z_max"], int(1.6 * params["z_max"]), dev)  # (Z, 1), (1, N)

    b = binding_energy(z, n, coeff)  # (Z, N) in MeV
    b_per_a = per_nucleon(b, z, n)  # (Z, N)
    mass, z_best, best = solve.most_stable(b, z, n)  # (K,) each
    z_line = stability_line(mass, coeff)  # (K,)

    a_peak, ba_peak = solve.peak(mass, best)
    print(f"B/A is largest at A = {a_peak:.0f}: {ba_peak:.3f} MeV per nucleon")
    print("nucleus    B formula   B measured   (MeV)")
    for name, zz, a, ba in tensors.MEASURED:
        print(f"{name:7s}  {b[zz - 1, a - zz].item():10.1f}  {ba * a:11.1f}")
    return plot.figure(z, n, b_per_a, mass, best, z_best, z_line, tensors.MEASURED, out_dir / "binding_energy.png")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/binding_energy/equation.py
r"""Physics → tensor translation.

The semi-empirical (Weizsäcker) mass formula treats the nucleus as a charged liquid drop. Its binding energy
for Z protons and N neutrons (A = Z + N) is

    B(Z, N) = a_V A - a_S A^{2/3} - a_C \frac{Z(Z-1)}{A^{1/3}} - a_A \frac{(N - Z)^2}{A} + δ(Z, N),
    δ = +a_P / \sqrt{A} (Z, N both even),   -a_P / \sqrt{A} (both odd),   0 (A odd).

On a grid Z_z (column vector) and N_n (row vector) every term broadcasts to a (Z, N) table B_{zn} —
there is no summed index, just one formula evaluated everywhere at once.

Setting ∂B/∂Z = 0 at fixed A (ignoring pairing and the −1 in Z(Z−1)) gives the valley of stability

    Z_0(A) = \frac{A}{2 + (a_C / 2 a_A) A^{2/3}}.
"""

import torch


def binding_energy(z: torch.Tensor, n: torch.Tensor, a: dict[str, float]) -> torch.Tensor:
    r"""B_{zn} in MeV.   z: (Z, 1), n: (1, N) → (Z, N) by broadcasting."""
    mass = z + n  # A_{zn}
    volume = a["a_V"] * mass
    surface = a["a_S"] * mass ** (2 / 3)
    coulomb = a["a_C"] * z * (z - 1) / mass ** (1 / 3)
    asymmetry = a["a_A"] * (n - z) ** 2 / mass
    return volume - surface - coulomb - asymmetry + pairing(z, n, a["a_P"])


def pairing(z: torch.Tensor, n: torch.Tensor, a_p: float) -> torch.Tensor:
    r"""δ_{zn} = ±a_P/\sqrt{A} for even-even / odd-odd nuclei, 0 for odd A.   → (Z, N)"""
    sign = ((z % 2 == 0) & (n % 2 == 0)).double() - ((z % 2 == 1) & (n % 2 == 1)).double()
    return sign * a_p / (z + n).sqrt()


def per_nucleon(b: torch.Tensor, z: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""B/A for every nucleus.   (Z, N) → (Z, N)"""
    return b / (z + n)


def stability_line(mass: torch.Tensor, a: dict[str, float]) -> torch.Tensor:
    r"""Z_0(A) = A / (2 + (a_C / 2a_A) A^{2/3}).   (K,) → (K,)"""
    return mass / (2 + a["a_C"] / (2 * a["a_A"]) * mass ** (2 / 3))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/binding_energy/tensors.py
"""Build the concrete tensors: the (Z, N) grid, the coefficients and a few measured nuclei."""

import torch


def grid(z_max: int, n_max: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Z as a column (Z, 1) and N as a row (1, N), so any formula in Z and N broadcasts to (Z, N)."""
    z = torch.arange(1, z_max + 1, dtype=torch.float64, device=device)[:, None]
    n = torch.arange(0, n_max + 1, dtype=torch.float64, device=device)[None, :]
    return z, n


def coefficients(params: dict) -> dict[str, float]:
    """The five liquid-drop coefficients in MeV."""
    return {key: params[key] for key in ("a_V", "a_S", "a_C", "a_A", "a_P")}


# (name, Z, A, measured B/A in MeV) — from the atomic mass evaluation, for comparison only
MEASURED = [
    ("²H", 1, 2, 1.112),
    ("⁴He", 2, 4, 7.074),
    ("¹²C", 6, 12, 7.680),
    ("¹⁶O", 8, 16, 7.976),
    ("⁵⁶Fe", 26, 56, 8.790),
    ("⁶²Ni", 28, 62, 8.795),
    ("¹²⁰Sn", 50, 120, 8.505),
    ("²⁰⁸Pb", 82, 208, 7.868),
    ("²³⁸U", 92, 238, 7.570),
]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/binding_energy/solve.py
"""Read off the physics from the (Z, N) table: the most bound isobar for each A and the B/A peak."""

import torch


def most_stable(b: torch.Tensor, z: torch.Tensor, n: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """For every mass number A, the Z with the largest binding energy among the isobars on the grid.

    Returns (A (K,), Z_best (K,), (B/A)_best (K,)). Isobars are the anti-diagonals Z + N = A of the table;
    cells with a different A are masked with −∞ before the max over Z.
    """
    mass = (z + n).long()  # (Z, N) mass number of each cell
    a_values = torch.arange(2, int(mass.max()) + 1, device=b.device)  # (K,)
    on_isobar = mass[None] == a_values[:, None, None]  # (K, Z, N)
    masked = torch.where(on_isobar, b[None], torch.tensor(-torch.inf, dtype=b.dtype, device=b.device))
    best_b, flat = masked.flatten(1).max(dim=1)  # best over all cells with this A
    z_best = z.flatten()[flat // b.shape[1]]
    keep = torch.isfinite(best_b) & (best_b > 0)
    a_values, z_best, best_b = a_values[keep].double(), z_best[keep], best_b[keep]
    return a_values, z_best, best_b / a_values


def peak(mass: torch.Tensor, b_per_a: torch.Tensor) -> tuple[float, float]:
    """(A, B/A) where the binding energy per nucleon is largest."""
    k = torch.argmax(b_per_a)
    return mass[k].item(), b_per_a[k].item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/binding_energy/plot.py
"""Two panels: B/A along the valley of stability, and a heatmap of B/A over the (N, Z) chart."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import BoundaryNorm

from problems._common.plotting import save_png


def figure(
    z: torch.Tensor,
    n: torch.Tensor,
    b_per_a: torch.Tensor,
    mass: torch.Tensor,
    best: torch.Tensor,
    z_best: torch.Tensor,
    z_line: torch.Tensor,
    measured: list,
    path: Path,
) -> Path:
    fig, (ax_curve, ax_map) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1, 1.15]})

    a_np, best_np = mass.cpu().numpy(), best.cpu().numpy()
    ax_curve.plot(a_np, best_np, color="#2563eb", lw=2, label="liquid-drop formula")
    offsets = {"⁵⁶Fe": (-22, 6), "⁶²Ni": (2, 6)}  # keep the two neighbouring labels apart
    for name, _z, a, ba in measured:
        ax_curve.plot(a, ba, "o", color="#dc2626", ms=5)
        offset = offsets.get(name, (4, -11))
        ax_curve.annotate(name, (a, ba), textcoords="offset points", xytext=offset, fontsize=8, color="#b91c1c")
    k = int(np.argmax(best_np))
    ax_curve.axvline(a_np[k], color="#f59e0b", lw=1, ls="--")
    ax_curve.text(a_np[k] + 4, 3.0, f"peak A = {a_np[k]:.0f}", color="#b45309", fontsize=9)
    ax_curve.plot([], [], "o", color="#dc2626", ms=5, label="measured")
    ax_curve.set_xlim(0, a_np.max())
    ax_curve.set_ylim(0, 9.5)
    ax_curve.set_xlabel("mass number A")
    ax_curve.set_ylabel("binding energy per nucleon B/A (MeV)")
    ax_curve.legend(loc="lower right", fontsize=8)

    ba = b_per_a.cpu().numpy()
    ba = np.where(ba > 0, ba, np.nan)  # unbound cells left blank
    levels = np.array([0.0, 4.0, 5.0, 6.0, 7.0, 7.5, 8.0, 8.5, 9.0])  # finer bins near the top
    cmap = plt.get_cmap("viridis", len(levels) - 1)
    image = ax_map.imshow(
        ba,
        origin="lower",
        extent=(n.min().item() - 0.5, n.max().item() + 0.5, z.min().item() - 0.5, z.max().item() + 0.5),
        cmap=cmap,
        norm=BoundaryNorm(levels, cmap.N, clip=True),
        aspect="auto",
        interpolation="nearest",
    )
    fig.colorbar(image, ax=ax_map, label="B/A (MeV)")
    z_np = z_best.cpu().numpy()
    ax_map.plot(a_np - z_np, z_np, color="#dc2626", lw=1.2, label="most bound isobar")
    z_line_np = z_line.cpu().numpy()
    ax_map.plot(a_np - z_line_np, z_line_np, color="white", lw=1, ls="--", label=r"$Z_0(A)$ formula")
    ax_map.plot([0, n.max().item()], [0, n.max().item()], color="#9ca3af", lw=1, ls=":", label="N = Z")
    ax_map.set_xlim(0, n.max().item())
    ax_map.set_ylim(0, z.max().item())
    ax_map.set_xlabel("neutrons N")
    ax_map.set_ylabel("protons Z")
    ax_map.grid(False)
    ax_map.legend(loc="upper left", fontsize=8)

    fig.tight_layout()
    return save_png(fig, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'a_V': 15.8,
    'a_S': 18.3,
    'a_C': 0.714,
    'a_A': 23.2,
    'a_P': 12.0,
    'z_max': 110
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.binding_energy")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.binding_energy.main")
problem_dir = Path("problems/nuclear/binding_energy")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
